# Detección de anomalías en paneles solares con YOLOv5

**Actividad 3 - Aprendizaje Profundo para Percepción y Control**

Este trabajo desarrolla un sistema de detección de objetos basado en YOLOv5 para localizar y clasificar anomalías visibles en paneles solares. El flujo experimental integra la preparación reproducible del entorno, la validación del conjunto de datos, el ajuste fino del modelo, la evaluación cuantitativa y la generación de inferencias sobre imágenes no utilizadas durante el entrenamiento.


## 1. Definición del problema

La inspección manual de instalaciones fotovoltaicas requiere tiempo y puede retrasar la identificación de condiciones que reducen la producción o demandan mantenimiento. El problema se formula como una tarea de detección de objetos con cuatro clases: panel cubierto (`cover`), grieta o rotura (`crack`), suciedad o polvo (`dust`) y panel en estado normal (`normal`).

El detector se plantea como una herramienta de apoyo para inspecciones realizadas mediante cámaras terrestres o drones. Su alcance se limita a anomalías visibles en imágenes RGB, por lo que no sustituye procedimientos técnicos capaces de identificar fallas eléctricas o térmicas.


## 2. Creación y preparación del dataset

El conjunto de datos procede de Roboflow Universe y se distribuye bajo licencia CC BY 4.0. Las anotaciones originales emplean el formato YOLO, donde cada objeto se representa mediante su clase y las coordenadas normalizadas de su caja delimitadora.

Para reducir la fuga de información, las variantes aumentadas derivadas de una misma imagen fuente se mantuvieron dentro de una única partición. La división reproducible utiliza la semilla 42 y asigna aproximadamente 70 % de las imágenes a entrenamiento, 20 % a validación y 10 % a prueba.

Fuente: https://universe.roboflow.com/workshop-pydqh/solar-panel-0swal-vrqxd-seo7f/dataset/1


### 2.1 Entorno de ejecución y reproducibilidad

El entorno experimental se construye automáticamente en Google Colab. El proceso comprueba la disponibilidad de GPU, clona el repositorio del proyecto en `/content/ProyectoYOLOv5`, obtiene una revisión fijada del repositorio oficial de YOLOv5, instala sus dependencias y descarga los pesos preentrenados `yolov5s.pt`.

La fijación de la revisión de YOLOv5 y la definición centralizada de rutas permiten reproducir el experimento sin depender de directorios privados o configuraciones específicas de un equipo local.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import urllib.request

REPO_URL = "https://github.com/Cesarin152/yolov5-paneles-solares.git"
YOLOV5_COMMIT = "88384748f0649754590fb96ec21506bd21c14718"
PROJECT_DIR = Path("/content/ProyectoYOLOv5")
YOLOV5_DIR = PROJECT_DIR / "vendor" / "yolov5"
DATA_DIR = PROJECT_DIR / "data" / "solar_panels"
BASE_WEIGHTS = PROJECT_DIR / "yolov5s.pt"
PYTHON_EXECUTABLE = Path(sys.executable)

def run(command, cwd=None):
    print("$", " ".join(map(str, command)))
    subprocess.run([str(item) for item in command], cwd=cwd, check=True)

# Comprobar la GPU antes de iniciar las descargas.
import torch
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: no hay GPU. Actívala en el menú de entorno de ejecución.")

# Clonar o reutilizar el proyecto.
if not PROJECT_DIR.exists():
    run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR])
elif not (PROJECT_DIR / ".git").is_dir():
    raise RuntimeError(f"{PROJECT_DIR} existe, pero no es un repositorio Git.")
else:
    print("Repositorio del proyecto ya disponible:", PROJECT_DIR)

# Clonar una versión reproducible y compatible con PyTorch reciente.
if not (YOLOV5_DIR / "train.py").is_file():
    if YOLOV5_DIR.exists():
        shutil.rmtree(YOLOV5_DIR)
    YOLOV5_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "init", YOLOV5_DIR])
    run([
        "git", "-C", YOLOV5_DIR, "remote", "add", "origin",
        "https://github.com/ultralytics/yolov5.git",
    ])
    run([
        "git", "-C", YOLOV5_DIR, "fetch", "--depth", "1",
        "origin", YOLOV5_COMMIT,
    ])
    run(["git", "-C", YOLOV5_DIR, "checkout", "--detach", "FETCH_HEAD"])
else:
    print("YOLOv5 ya disponible:", YOLOV5_DIR)

run([
    PYTHON_EXECUTABLE, "-m", "pip", "install", "-q",
    "-r", YOLOV5_DIR / "requirements.txt",
])

if not BASE_WEIGHTS.is_file():
    weights_url = (
        "https://github.com/ultralytics/yolov5/releases/download/"
        "v7.0/yolov5s.pt"
    )
    print("Descargando pesos:", weights_url)
    urllib.request.urlretrieve(weights_url, BASE_WEIGHTS)
else:
    print("Pesos ya disponibles:", BASE_WEIGHTS)

os.chdir(PROJECT_DIR)
print("Directorio activo:", Path.cwd())
print("Dataset:", DATA_DIR)


### 2.2 Configuración del conjunto de datos

El repositorio contiene las imágenes y etiquetas organizadas en las particiones de entrenamiento, validación y prueba. A partir de `/content/ProyectoYOLOv5` se construyen las rutas del experimento y se genera un archivo YAML compatible con YOLOv5, en el que se declaran las cuatro clases y la ubicación de cada partición.


In [ ]:
# Validación de archivos y generación del YAML con rutas de Colab.
required_paths = [
    DATA_DIR / "images" / "train",
    DATA_DIR / "images" / "val",
    DATA_DIR / "images" / "test",
    DATA_DIR / "labels" / "train",
    DATA_DIR / "labels" / "val",
    DATA_DIR / "labels" / "test",
    PROJECT_DIR / "scripts" / "validate_dataset.py",
    YOLOV5_DIR / "train.py",
    YOLOV5_DIR / "val.py",
    YOLOV5_DIR / "detect.py",
    BASE_WEIGHTS,
]
missing_paths = [path for path in required_paths if not path.exists()]
assert not missing_paths, "Faltan archivos requeridos:\n" + "\n".join(map(str, missing_paths))

RUNTIME_YAML = PROJECT_DIR / "solar_panels_runtime.yaml"
RUNTIME_YAML.write_text(
    f"""path: {DATA_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

nc: 4
names: [cover, crack, dust, normal]
""",
    encoding="utf-8",
)
print(RUNTIME_YAML.read_text(encoding="utf-8"))


### 2.3 Verificación del conjunto de datos

La verificación comprueba la correspondencia entre imágenes y etiquetas, la validez de las anotaciones y la distribución de objetos por clase. Esta etapa permite detectar archivos faltantes, identificadores de clase incorrectos o coordenadas fuera del intervalo normalizado antes del entrenamiento.


In [ ]:
import json
import subprocess
import sys

report_path = PROJECT_DIR / "reports" / "dataset_report.json"
subprocess.run(
    [sys.executable, str(PROJECT_DIR / "scripts" / "validate_dataset.py"),
     "--dataset", str(DATA_DIR), "--output", str(report_path)],
    check=True,
)
report = json.loads(report_path.read_text(encoding="utf-8"))
report


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import random

class_names = ["cover", "crack", "dust", "normal"]
colors = ["#ff9800", "#f44336", "#795548", "#4caf50"]

def show_annotated(image_path):
    label_path = Path(str(image_path).replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)).with_suffix(".txt")
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    for line in label_path.read_text().splitlines():
        cls, xc, yc, bw, bh = map(float, line.split())
        cls = int(cls)
        x1, y1 = (xc - bw / 2) * width, (yc - bh / 2) * height
        x2, y2 = (xc + bw / 2) * width, (yc + bh / 2) * height
        draw.rectangle((x1, y1, x2, y2), outline=colors[cls], width=3)
        draw.text((x1 + 3, y1 + 3), class_names[cls], fill=colors[cls])
    return image

samples = random.Random(42).sample(list((DATA_DIR / "images" / "train").glob("*")), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, path in zip(axes.flat, samples):
    ax.imshow(show_annotated(path))
    ax.set_title(path.name[:35])
    ax.axis("off")
plt.tight_layout()


## 3. Proceso de entrenamiento del modelo

Se utiliza YOLOv5s inicializado con pesos preentrenados en COCO. Esta arquitectura ofrece un equilibrio adecuado entre capacidad de detección, velocidad y consumo de memoria para el alcance del experimento.

El ajuste fino se configura durante 50 épocas, con imágenes de 640 píxeles y un tamaño de lote de 16. El entrenamiento emplea las particiones definidas en el archivo YAML, conserva los mejores pesos según el desempeño en validación y almacena métricas, curvas y matrices de confusión para el análisis posterior.


In [ ]:
# Comprobación posterior a la instalación.
required_modules = [
    "torch", "torchvision", "cv2", "numpy", "pandas",
    "matplotlib", "yaml", "PIL",
]
missing_modules = []
for module_name in required_modules:
    try:
        __import__(module_name)
    except ImportError:
        missing_modules.append(module_name)

assert not missing_modules, "Faltan dependencias: " + ", ".join(missing_modules)
print("Dependencias verificadas correctamente.")


In [ ]:
# Confirmación del dispositivo que utilizará YOLOv5.
import torch

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("Dispositivo seleccionado:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("El entrenamiento funcionará en CPU, pero será considerablemente más lento.")


In [ ]:
# Parámetros editables del entrenamiento completo.
IMG_SIZE = 640
BATCH_SIZE = 16       # Reduce a 8 o 4 si la GPU se queda sin memoria.
EPOCHS = 50
RUN_NAME = "solar_panels_yolov5s"
RUN_DIR = PROJECT_DIR / "runs" / "train" / RUN_NAME

train_command = [
    PYTHON_EXECUTABLE, YOLOV5_DIR / "train.py",
    "--img", IMG_SIZE,
    "--batch", BATCH_SIZE,
    "--epochs", EPOCHS,
    "--data", RUNTIME_YAML,
    "--weights", BASE_WEIGHTS,
    "--project", PROJECT_DIR / "runs" / "train",
    "--name", RUN_NAME,
    "--device", DEVICE,
    "--cache",
    "--exist-ok",
]
run(train_command, cwd=YOLOV5_DIR)


## 4. Evaluación y resultados del modelo

El desempeño se analiza mediante precisión, recall, mAP@0.5 y mAP@0.5:0.95. Las curvas generadas durante el entrenamiento permiten observar la convergencia de las pérdidas y la evolución de las métricas, mientras que la matriz de confusión muestra el comportamiento específico de cada clase.

Además de la validación realizada durante el ajuste, los mejores pesos se evalúan sobre la partición de prueba, reservada para estimar la capacidad de generalización del detector.


In [ ]:
from IPython.display import display
import pandas as pd

results_path = RUN_DIR / "results.csv"
assert results_path.is_file(), "Ejecuta primero la celda de entrenamiento."
results = pd.read_csv(results_path)
results.columns = results.columns.str.strip()
display(results.tail())

metric_columns = [
    column for column in results.columns
    if any(key in column for key in ("precision", "recall", "mAP_0.5", "mAP_0.5:0.95"))
]
display(results[metric_columns].tail(1))


In [ ]:
from IPython.display import Image as DisplayImage, display

for filename in ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png"]:
    path = RUN_DIR / filename
    if path.exists():
        print(filename)
        display(DisplayImage(filename=str(path)))


In [ ]:
# Evaluación independiente sobre el conjunto de prueba.
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
assert BEST_WEIGHTS.exists(), f"No se encontraron los pesos: {BEST_WEIGHTS}"

validation_command = [
    PYTHON_EXECUTABLE, YOLOV5_DIR / "val.py",
    "--weights", BEST_WEIGHTS,
    "--data", RUNTIME_YAML,
    "--img", IMG_SIZE,
    "--task", "test",
    "--project", PROJECT_DIR / "runs" / "val",
    "--name", "test_evaluation",
    "--device", DEVICE,
    "--save-txt",
    "--save-conf",
    "--exist-ok",
]
run(validation_command, cwd=YOLOV5_DIR)


### 4.1 Criterios de interpretación

La precisión representa la proporción de detecciones positivas que corresponde a objetos correctamente identificados, mientras que el recall refleja la proporción de objetos reales recuperados por el modelo. Las métricas mAP resumen la precisión promedio para distintos niveles de confianza y solapamiento.

La diferencia entre mAP@0.5 y mAP@0.5:0.95 permite valorar cuánto disminuye el desempeño cuando se exige una localización más precisa. El análisis por clase resulta especialmente relevante para `crack` y `dust`, debido a que los falsos negativos en estas anomalías podrían retrasar intervenciones de mantenimiento.

La idoneidad del sistema no depende únicamente del resultado global. También se consideran la estabilidad de las curvas, las confusiones entre clases y el equilibrio entre falsos positivos y falsos negativos.


## 5. Inferencia del modelo

La inferencia se realiza con los mejores pesos obtenidos durante el entrenamiento y utiliza las imágenes de la partición de prueba. Para cada detección, YOLOv5 genera una caja delimitadora, la clase predicha y su nivel de confianza.

Las predicciones se almacenan como imágenes anotadas y archivos de texto, lo que facilita tanto la inspección visual como el análisis posterior de los resultados.


In [ ]:
# Inferencia sobre imágenes reservadas del conjunto de prueba.
TEST_IMAGES = DATA_DIR / "images" / "test"
DETECTION_RUN = PROJECT_DIR / "runs" / "detect" / "test_predictions"

detection_command = [
    PYTHON_EXECUTABLE, YOLOV5_DIR / "detect.py",
    "--weights", BEST_WEIGHTS,
    "--img", IMG_SIZE,
    "--conf", 0.25,
    "--source", TEST_IMAGES,
    "--project", PROJECT_DIR / "runs" / "detect",
    "--name", "test_predictions",
    "--device", DEVICE,
    "--save-txt",
    "--save-conf",
    "--exist-ok",
]
run(detection_command, cwd=YOLOV5_DIR)


In [ ]:
from IPython.display import Image as DisplayImage, display

prediction_files = [
    path
    for path in DETECTION_RUN.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

for path in sorted(prediction_files)[:12]:
    display(DisplayImage(filename=str(path), width=700))


### 5.1 Análisis cualitativo de las inferencias

La inspección cualitativa permite estudiar detecciones correctas, falsos positivos, objetos omitidos y confusiones entre categorías. Factores como iluminación, perspectiva, distancia, oclusión y tamaño de la anomalía pueden modificar la confianza y la precisión de las cajas delimitadoras.

Los errores observados orientan posibles mejoras del sistema, entre ellas la incorporación de más imágenes originales, la revisión de etiquetas, el balance de clases y la captura de escenarios fotovoltaicos más diversos.


## 6. Conclusiones

El flujo desarrollado cubre de forma reproducible las etapas principales de un proyecto de detección de objetos: validación de datos, transferencia de aprendizaje, entrenamiento, evaluación independiente e inferencia. La separación entre entrenamiento, validación y prueba permite analizar el comportamiento del modelo sobre imágenes no utilizadas para ajustar sus parámetros.

El conjunto de datos incluye aumentos generados previamente. Para reducir una estimación excesivamente optimista, las variantes asociadas a una misma imagen fuente se agruparon dentro de una única partición. Aun con esta precaución, la aplicación en un entorno real requeriría ampliar la diversidad de condiciones de captura y complementar la inspección RGB con métodos adecuados para fallas no visibles.


## 7. Referencias

- Ultralytics. Repositorio oficial de YOLOv5: https://github.com/ultralytics/yolov5
- Dataset Solar Panel, Roboflow Universe, licencia CC BY 4.0: https://universe.roboflow.com/workshop-pydqh/solar-panel-0swal-vrqxd-seo7f/dataset/1
- Jocher, G. et al. YOLOv5 documentation and source code.
